# 07 — Merged Analytical Panels

**Goal:** Construct the two final analytical datasets the regressions will run on:
(A) firm-month panel for asset-pricing tests, and (B) firm-year panel for
operating-performance tests.

**Inputs:**
- `sp500_universe_with_gvkey.parquet` (6,535 firm-year universe)
- `sentiment_panel.parquet` (8,069 firm-year sentiment scores)
- `returns_panel.parquet` (98,978 firm-month returns)
- `fundamentals_panel.parquet` (9,048 firm-year fundamentals)
- `factors.parquet` (156 monthly factor returns)

**Outputs:**
- `data/panel_monthly.parquet` — firm-month panel for H1/H2a
- `data/panel_yearly.parquet` — firm-year panel for H2b

**Lag convention:** Sentiment scores from calendar year t-1 are attached to
outcomes in year t. For the monthly panel, all twelve months of year t are
matched to the same lagged sentiment value (constructed from reviews through
Dec 31 of year t-1). For the yearly panel, the same one-year lag is applied
to fundamentals.

**Membership restriction (fix 2026-07-02):** Both panels are inner-joined to
the point-in-time universe on (gvkey, year): a firm contributes outcome rows
only for the years it was actually an S&P 500 member. Sentiment lags may come
from non-membership years (the signal needs no membership); outcomes may not.
See `docs/membership_fix_memo_2026-07-02.md`.

**Sample window:** Outcomes observed 2013–2024 (requires year-2012 sentiment
as the first available lag).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"

# Load all five constructed panels
universe = pd.read_parquet(DATA_PROCESSED / "sp500_universe_with_gvkey.parquet")
sentiment = pd.read_parquet(DATA_PROCESSED / "sentiment_panel.parquet")
returns = pd.read_parquet(DATA_PROCESSED / "returns_panel.parquet")
fundamentals = pd.read_parquet(DATA_PROCESSED / "fundamentals_panel.parquet")
factors = pd.read_parquet(DATA_PROCESSED / "factors.parquet")

# Normalize gvkey dtype to string across all panels (so merges work cleanly)
for df, name in [(universe, "universe"), (sentiment, "sentiment"),
                  (returns, "returns"), (fundamentals, "fundamentals")]:
    df["gvkey"] = df["gvkey"].astype(str)

print("Panel sizes:")
print(f"  Universe:     {len(universe):>7,} rows")
print(f"  Sentiment:    {len(sentiment):>7,} rows")
print(f"  Returns:      {len(returns):>7,} rows")
print(f"  Fundamentals: {len(fundamentals):>7,} rows")
print(f"  Factors:      {len(factors):>7,} rows")
print()

print("Year coverage by panel:")
for df, name, col in [
    (universe, "universe", "year"),
    (sentiment, "sentiment", "year"),
    (returns, "returns", "year"),
    (fundamentals, "fundamentals", "fyear"),
]:
    print(f"  {name:13s}: {df[col].min()} to {df[col].max()}")

Panel sizes:
  Universe:       6,535 rows
  Sentiment:      8,069 rows
  Returns:       98,978 rows
  Fundamentals:   9,048 rows
  Factors:          156 rows

Year coverage by panel:
  universe     : 2012 to 2024
  sentiment    : 2012 to 2024
  returns      : 2012 to 2024
  fundamentals : 2011 to 2024


In [2]:
# Create a "lag" version of the sentiment panel where each row is reattributed
# from its actual year to year+1 (the year the lagged sentiment "applies" to)
sentiment_lag = sentiment.copy()
sentiment_lag["year"] = sentiment_lag["year"] + 1
sentiment_lag = sentiment_lag.rename(columns={
    "sentiment_overall": "sentiment_overall_lag",
    "sentiment_wlb": "sentiment_wlb_lag",
    "sentiment_comp": "sentiment_comp_lag",
    "sentiment_career": "sentiment_career_lag",
    "sentiment_culture": "sentiment_culture_lag",
    "sentiment_leadership": "sentiment_leadership_lag",
    "n_reviews": "n_reviews_lag",
})

print(f"Lagged sentiment panel: {len(sentiment_lag):,} rows")
print(f"Year range (now represents 'applies to year'): {sentiment_lag['year'].min()} to {sentiment_lag['year'].max()}")
print()
print("Sample rows:")
print(sentiment_lag.head())

Lagged sentiment panel: 8,069 rows
Year range (now represents 'applies to year'): 2013 to 2025

Sample rows:
    gvkey  year  n_reviews_lag  sentiment_overall_lag  sentiment_wlb_lag  sentiment_comp_lag  sentiment_career_lag  sentiment_culture_lag  sentiment_leadership_lag
0  001045  2013            112               2.669643           3.009259            2.574074              2.613636               2.549296                  2.191589
1  001045  2014            125                  3.008           2.929204            2.675439              2.868421               2.814159                  2.428571
2  001045  2015            177                3.20904           3.095541            3.089172              3.134615                3.11465                  2.574194
3  001045  2016            389               3.393316           3.106195            3.247059              3.312684                3.16568                  2.768997
4  001045  2017            412               3.667476            3.3081

In [3]:
# Start with returns and attach lagged sentiment by gvkey-year
panel_monthly = returns.merge(
    sentiment_lag,
    on=["gvkey", "year"],
    how="left"
)

# Attach factor data on date
panel_monthly = panel_monthly.merge(factors, on=["date", "year", "month"], how="left")

# Construct excess return (return minus risk-free rate) — used in factor regressions
panel_monthly["excess_ret"] = panel_monthly["ret"] - panel_monthly["rf"]

# Reorder columns for clarity
col_order = [
    "gvkey", "permno", "date", "year", "month",
    "ret", "ret_ex_dividend", "excess_ret", "prc", "shrout", "me",
    "sentiment_overall_lag", "n_reviews_lag",
    "sentiment_wlb_lag", "sentiment_comp_lag", "sentiment_career_lag",
    "sentiment_culture_lag", "sentiment_leadership_lag",
    "mkt_rf", "smb", "hml", "rmw", "cma", "mom", "rf",
]
panel_monthly = panel_monthly[col_order].sort_values(["gvkey", "date"]).reset_index(drop=True)

print(f"Panel A (monthly) shape: {panel_monthly.shape}")
print()

# Diagnostic: how many firm-months have valid lagged sentiment?
n_total = len(panel_monthly)
n_with_sentiment = panel_monthly["sentiment_overall_lag"].notna().sum()
n_with_factors = panel_monthly["mkt_rf"].notna().sum()
print(f"Firm-month rows: {n_total:,}")
print(f"  With lagged sentiment: {n_with_sentiment:,} ({n_with_sentiment/n_total:.1%})")
print(f"  With factor data: {n_with_factors:,} ({n_with_factors/n_total:.1%})")
print()

# Cove

Panel A (monthly) shape: (98978, 25)

Firm-month rows: 98,978
  With lagged sentiment: 82,190 (83.0%)
  With factor data: 69,796 (70.5%)



In [4]:
# Diagnose the factor merge failure
print("returns_panel dtypes:")
print(returns[["date", "year", "month"]].dtypes)
print()
print("factors dtypes:")
print(factors[["date", "year", "month"]].dtypes)
print()
print("Sample dates from each:")
print(f"returns first date: {returns['date'].iloc[0]} (type: {type(returns['date'].iloc[0]).__name__})")
print(f"factors first date: {factors['date'].iloc[0]} (type: {type(factors['date'].iloc[0]).__name__})")
print()
print(f"returns date range: {returns['date'].min()} to {returns['date'].max()}")
print(f"factors date range: {factors['date'].min()} to {factors['date'].max()}")
print()

# Check whether the dates actually match
returns_dates = set(returns["date"].unique())
factor_dates = set(factors["date"].unique())
print(f"Unique returns dates: {len(returns_dates)}")
print(f"Unique factor dates: {len(factor_dates)}")
print(f"Intersection: {len(returns_dates & factor_dates)}")
print(f"Dates in returns but not factors: {len(returns_dates - factor_dates)}")
print(f"Dates in factors but not returns: {len(factor_dates - returns_dates)}")

returns_panel dtypes:
date     datetime64[ns]
year              int32
month             int32
dtype: object

factors dtypes:
date     datetime64[ns]
year              int32
month             int32
dtype: object

Sample dates from each:
returns first date: 2013-12-31 00:00:00 (type: Timestamp)
factors first date: 2012-01-31 00:00:00 (type: Timestamp)

returns date range: 2012-01-31 00:00:00 to 2024-12-31 00:00:00
factors date range: 2012-01-31 00:00:00 to 2024-12-31 00:00:00

Unique returns dates: 156
Unique factor dates: 156
Intersection: 110
Dates in returns but not factors: 46
Dates in factors but not returns: 46


In [5]:
# Drop the previously-merged factor columns (they're mostly NaN due to the date mismatch)
factor_cols = ["mkt_rf", "smb", "hml", "rmw", "cma", "mom", "rf", "excess_ret"]
panel_monthly = panel_monthly.drop(columns=[c for c in factor_cols if c in panel_monthly.columns])

# Re-attach factor data — this time on (year, month) only, not on date
panel_monthly = panel_monthly.merge(
    factors.drop(columns=["date"]),  # drop date to avoid duplicate-column naming
    on=["year", "month"],
    how="left"
)

# Construct excess return
panel_monthly["excess_ret"] = panel_monthly["ret"] - panel_monthly["rf"]

# Reorder
col_order = [
    "gvkey", "permno", "date", "year", "month",
    "ret", "ret_ex_dividend", "excess_ret", "prc", "shrout", "me",
    "sentiment_overall_lag", "n_reviews_lag",
    "sentiment_wlb_lag", "sentiment_comp_lag", "sentiment_career_lag",
    "sentiment_culture_lag", "sentiment_leadership_lag",
    "mkt_rf", "smb", "hml", "rmw", "cma", "mom", "rf",
]
panel_monthly = panel_monthly[col_order].sort_values(["gvkey", "date"]).reset_index(drop=True)

print(f"Panel A (monthly) shape: {panel_monthly.shape}")
print()

n_total = len(panel_monthly)
n_with_sentiment = panel_monthly["sentiment_overall_lag"].notna().sum()
n_with_factors = panel_monthly["mkt_rf"].notna().sum()
print(f"Firm-month rows: {n_total:,}")
print(f"  With lagged sentiment: {n_with_sentiment:,} ({n_with_sentiment/n_total:.1%})")
print(f"  With factor data: {n_with_factors:,} ({n_with_factors/n_total:.1%})")
print()

print("Coverage by year:")
coverage = panel_monthly.groupby("year").agg(
    total_firm_months=("gvkey", "count"),
    with_sentiment_lag=("sentiment_overall_lag", lambda x: x.notna().sum()),
    with_factors=("mkt_rf", lambda x: x.notna().sum()),
).assign(
    sent_pct=lambda d: d["with_sentiment_lag"] / d["total_firm_months"],
    fact_pct=lambda d: d["with_factors"] / d["total_firm_months"]
)
print(coverage.round(3))

Panel A (monthly) shape: (98978, 25)

Firm-month rows: 98,978
  With lagged sentiment: 82,190 (83.0%)
  With factor data: 98,978 (100.0%)

Coverage by year:
      total_firm_months  with_sentiment_lag  with_factors  sent_pct  fact_pct
year                                                                         
2012               7883                 0.0        7883.0       0.0       1.0
2013               7987              6611.0        7987.0     0.828       1.0
2014               8009              6752.0        8009.0     0.843       1.0
2015               7986              6885.0        7986.0     0.862       1.0
2016               7799              6946.0        7799.0     0.891       1.0
2017               7722              6903.0        7722.0     0.894       1.0
2018               7666              6936.0        7666.0     0.905       1.0
2019               7569              6887.0        7569.0      0.91       1.0
2020               7412              6881.0        7412.0     0

In [6]:
# Filter to rows with both sentiment and factor data
panel_monthly_clean = panel_monthly[
    panel_monthly["sentiment_overall_lag"].notna() &
    panel_monthly["mkt_rf"].notna()
].copy()

# Also explicitly restrict to 2013-2024 (sample window)
panel_monthly_clean = panel_monthly_clean[
    (panel_monthly_clean["year"] >= 2013) &
    (panel_monthly_clean["year"] <= 2024)
].copy().reset_index(drop=True)

# MEMBERSHIP FIX 2026-07-02 (see docs/membership_fix_memo_2026-07-02.md):
# a firm-month enters the panel only if the firm was an S&P 500 member in that
# calendar year. Without this inner join, pre-inclusion and post-exit months of
# ever-members leak in (~21% of rows), contradicting the point-in-time design.
member_pairs = universe[["gvkey", "year"]].drop_duplicates().copy()
member_pairs["year"] = member_pairs["year"].astype(panel_monthly_clean["year"].dtype)
n_before = len(panel_monthly_clean)
panel_monthly_clean = panel_monthly_clean.merge(
    member_pairs, on=["gvkey", "year"], how="inner"
).reset_index(drop=True)
print(f"Membership filter: {n_before:,} -> {len(panel_monthly_clean):,} firm-months "
      f"({n_before - len(panel_monthly_clean):,} non-membership rows removed)")
print()

print(f"Panel A — final analytical sample: {len(panel_monthly_clean):,} firm-months")
print(f"Unique firms: {panel_monthly_clean['gvkey'].nunique()}")
print(f"Year range: {panel_monthly_clean['year'].min()} to {panel_monthly_clean['year'].max()}")
print()

# Distribution of the key sentiment IV
print("Lagged sentiment distribution:")
print(panel_monthly_clean["sentiment_overall_lag"].describe().round(3))
print()

# Distribution of returns (should match what we saw in returns_panel)
print("Excess returns distribution:")
print(panel_monthly_clean["excess_ret"].describe().round(4))

Membership filter: 82,190 -> 65,291 firm-months (16,899 non-membership rows removed)

Panel A — final analytical sample: 65,291 firm-months
Unique firms: 636
Year range: 2013 to 2024

Lagged sentiment distribution:
count    65291.0
mean       3.516
std        0.475
min          1.0
25%         3.25
50%        3.545
75%        3.824
max          5.0
Name: sentiment_overall_lag, dtype: Float64

Excess returns distribution:
count    65275.0
mean      0.0108
std       0.0846
min      -0.7071
25%      -0.0371
50%       0.0107
75%        0.057
max       0.8602
Name: excess_ret, dtype: Float64


In [7]:
output_path = DATA_PROCESSED / "panel_monthly.parquet"
panel_monthly_clean.to_parquet(output_path, index=False)

check = pd.read_parquet(output_path)
print(f"Saved {len(check):,} rows to {output_path.name}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Saved 65,291 rows to panel_monthly.parquet
File size: 3237.1 KB


In [8]:
# Start with fundamentals, attach lagged sentiment on (gvkey, year)
# Note: fundamentals uses 'fyear' (fiscal year) — we treat it as the year for matching purposes
panel_yearly = fundamentals.merge(
    sentiment_lag.rename(columns={"year": "fyear"}),
    on=["gvkey", "fyear"],
    how="left"
)

# Reorder columns
col_order = [
    "gvkey", "fyear", "datadate", "sich",
    # Operating-performance outcomes
    "roa", "operating_margin", "gross_margin", "sales_growth",
    # Raw fundamentals
    "ni", "at", "revt", "cogs", "oibdp", "lt", "seq",
    # Controls
    "log_at", "leverage",
    # Lagged sentiment (the IV)
    "sentiment_overall_lag", "n_reviews_lag",
    "sentiment_wlb_lag", "sentiment_comp_lag", "sentiment_career_lag",
    "sentiment_culture_lag", "sentiment_leadership_lag",
]
panel_yearly = panel_yearly[col_order].sort_values(["gvkey", "fyear"]).reset_index(drop=True)

print(f"Panel B (yearly) shape: {panel_yearly.shape}")
print()

# Coverage diagnostics
n_total = len(panel_yearly)
n_with_sentiment = panel_yearly["sentiment_overall_lag"].notna().sum()
n_with_roa = panel_yearly["roa"].notna().sum()
n_with_both = (panel_yearly["sentiment_overall_lag"].notna() &
               panel_yearly["roa"].notna()).sum()
print(f"Firm-year rows: {n_total:,}")
print(f"  With lagged sentiment: {n_with_sentiment:,} ({n_with_sentiment/n_total:.1%})")
print(f"  With ROA: {n_with_roa:,} ({n_with_roa/n_total:.1%})")
print(f"  With both (before membership filter): {n_with_both:,} ({n_with_both/n_total:.1%})")
print()

print("Coverage by year:")
coverage = panel_yearly.groupby("fyear").agg(
    total=("gvkey", "count"),
    with_sent=("sentiment_overall_lag", lambda x: x.notna().sum()),
    with_roa=("roa", lambda x: x.notna().sum()),
).assign(both_pct=lambda d: (d["with_sent"] * d["with_roa"]).clip(upper=d[["with_sent","with_roa"]].min(axis=1)) / d["total"])
print(coverage.round(3))

Panel B (yearly) shape: (9048, 24)

Firm-year rows: 9,048
  With lagged sentiment: 6,877 (76.0%)
  With ROA: 9,014 (99.6%)
  With both (before membership filter): 6,864 (75.9%)

Coverage by year:
       total  with_sent  with_roa  both_pct
fyear                                      
2011     678        0.0     676.0       0.0
2012     684        0.0     681.0       0.0
2013     688      563.0     685.0     0.818
2014     686      571.0     684.0     0.832
2015     671      575.0     670.0     0.857
2016     667      586.0     658.0     0.879
2017     654      580.0     649.0     0.887
2018     644      580.0     643.0     0.901
2019     634      576.0     632.0     0.909
2020     623      574.0     621.0     0.921
2021     619      573.0     615.0     0.926
2022     609      572.0     609.0     0.939
2023     600      566.0     600.0     0.943
2024     591      561.0     591.0     0.949


In [9]:
# Filter to rows usable for H2b: must have both lagged sentiment and at least ROA
panel_yearly_clean = panel_yearly[
    panel_yearly["sentiment_overall_lag"].notna() &
    panel_yearly["roa"].notna()
].copy()

# Restrict to sample window
panel_yearly_clean = panel_yearly_clean[
    (panel_yearly_clean["fyear"] >= 2013) &
    (panel_yearly_clean["fyear"] <= 2024)
].copy().reset_index(drop=True)

# MEMBERSHIP FIX 2026-07-02 (see docs/membership_fix_memo_2026-07-02.md):
# a firm-year enters only if the firm was an S&P 500 member in that year
# (fyear matched to the membership calendar year, consistent with the lag
# convention above).
member_pairs_fy = universe[["gvkey", "year"]].drop_duplicates().rename(columns={"year": "fyear"}).copy()
member_pairs_fy["fyear"] = member_pairs_fy["fyear"].astype(panel_yearly_clean["fyear"].dtype)
n_before = len(panel_yearly_clean)
panel_yearly_clean = panel_yearly_clean.merge(
    member_pairs_fy, on=["gvkey", "fyear"], how="inner"
).reset_index(drop=True)
print(f"Membership filter: {n_before:,} -> {len(panel_yearly_clean):,} firm-years "
      f"({n_before - len(panel_yearly_clean):,} non-membership rows removed)")
print()

print(f"Panel B — final analytical sample: {len(panel_yearly_clean):,} firm-years")
print(f"Unique firms: {panel_yearly_clean['gvkey'].nunique()}")
print(f"Year range: {panel_yearly_clean['fyear'].min()} to {panel_yearly_clean['fyear'].max()}")
print()

# Sanity check the outcome variables
print("Outcome variable distributions:")
print(panel_yearly_clean[["roa", "operating_margin", "gross_margin", "sales_growth"]].describe().round(3))
print()

print("Lagged sentiment distribution:")
print(panel_yearly_clean["sentiment_overall_lag"].describe().round(3))

# Save
output_path = DATA_PROCESSED / "panel_yearly.parquet"
panel_yearly_clean.to_parquet(output_path, index=False)

check = pd.read_parquet(output_path)
print()
print(f"Saved {len(check):,} rows to {output_path.name}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Membership filter: 6,864 -> 5,434 firm-years (1,430 non-membership rows removed)

Panel B — final analytical sample: 5,434 firm-years
Unique firms: 635
Year range: 2013 to 2024

Outcome variable distributions:
          roa  operating_margin  gross_margin  sales_growth
count  5434.0            5182.0        5434.0        5432.0
mean    0.066             0.252         0.448         0.079
std     0.076             0.173         0.236         0.381
min    -0.846            -2.874        -2.505        -0.839
25%     0.025             0.151         0.285        -0.007
50%     0.055             0.231         0.418         0.051
75%     0.099             0.344         0.607         0.122
max     0.653             0.816         1.108        21.991

Lagged sentiment distribution:
count    5434.0
mean      3.517
std       0.474
min         1.0
25%       3.251
50%       3.546
75%       3.825
max         5.0
Name: sentiment_overall_lag, dtype: Float64



Saved 5,434 rows to panel_yearly.parquet
File size: 807.2 KB
